# Database Paradigm Benchmark

**Runs on Colab or local Jupyter.** No GPU is required for the retrieval-scoring
portion of this notebook (query embedding for the two OpenAI models is a live API
call; `text-embedding-3-small`/`large` never touch a local model). A GPU is not
required for the open-source model either, since `e5-base-v2` on query-sized
batches (hundreds of texts) runs fine on CPU.

**Purpose:** Compare three database paradigms, PostgreSQL/pgvector (via Supabase),
MongoDB Atlas Vector Search, and Qdrant, on retrieval quality and latency, holding
the embedding model fixed per workload. This is the paper's core contribution: the
embedding model benchmark (prior notebook) answered "which embedding model", this
notebook answers "which database paradigm".

**Why the embedding model is fixed per workload, not globally:** The embedding
model benchmark found no single model winning every workload on the abstracts
slice. Per the abstracts-slice results:

| Workload | Winning model | Rationale |
|---|---|---|
| W1 (short-context factual) | `intfloat/e5-base-v2` | Wins every metric (MRR 0.590, Recall@1 0.538) |
| W2 (multi-hop reasoning) | `text-embedding-3-small` | Wins MRR (0.703) and Recall@1 (0.323), the primary ranking metrics |
| W3 (semantic similarity) | `text-embedding-3-large` | Wins every metric clearly (MRR 0.735, Recall@1 0.672) |

Each workload's chosen model is held fixed across all three database paradigms and
both content types (abstracts, pdf_chunks), so any performance difference observed
between pgvector, MongoDB, and Qdrant for a given workload is attributable to the
storage/indexing layer, not to embedding quality. This mirrors the control-variable
logic used in the embedding model benchmark itself.

**Embeddings are NOT pre-ingested.** This notebook performs both stages:

1. **Ingestion** — load each workload's fixed-model chunk embeddings from the HF
   dataset and upsert them into all three databases (idempotent: skips chunks
   already present).
2. **Querying** — embed each workload's queries with that workload's fixed model,
   run the retrieval query against each database, and score with the same `ranx`
   metrics used in the embedding model benchmark.

**Resumability.** Results are checkpointed at (database, workload, content_type,
category) granularity, the finest level available, and pushed to the private HF
dataset immediately after each unit completes. Re-running this notebook after an
interruption skips every unit already present on HF, so a crash never costs more
than the one in-flight unit.

**Data source:** `Sakhiur/empirical-rag-paradigm-benchmark` (chunk embeddings +
query CSVs), same private HF dataset used throughout the pipeline.


## Setup
Runs on Colab or local Jupyter unchanged. `%pip` (rather than `!pip`) targets the
kernel actually running this notebook, matching the convention used in the
embedding model benchmark and generation notebooks.


In [1]:
%pip install -q ranx rich huggingface_hub pandas numpy scikit-learn matplotlib sentence-transformers openai python-dotenv psycopg2-binary pymongo qdrant-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import string
import time
import gc
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ranx import Qrels, Run, compare
from ranx.data_structures.report import Report

from rich.box import HEAVY
from rich.text import Text
from rich.table import Table
from rich.console import Console

from huggingface_hub import hf_hub_download, snapshot_download, HfApi


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = running_in_colab()

print(f"Environment: {'Colab' if IN_COLAB else 'local Jupyter'}")

Environment: local Jupyter


### API keys, DB credentials, and HF token
Same pattern as the embedding model benchmark: Colab pulls from Colab Secrets,
local Jupyter pulls from the shell environment or a `.env` file.

Three database connection variables are required in addition to `HF_TOKEN` and
`OPENAI_API_KEY` (needed for W2/W3 query embedding and, for W1, only if you want
to cross-check against an OpenAI model; W1 itself uses the local `e5-base-v2`
model so no API key is strictly required for W1 alone):

- `SUPABASE_URL` — Postgres/pgvector connection string (Supabase)
- `MONGODB_URI` — MongoDB Atlas connection string
- `QDRANT_URL`, `QDRANT_API_KEY` — Qdrant Cloud endpoint + key


In [4]:
REQUIRED_ENV_VARS = (
    "HF_TOKEN",
    "OPENAI_API_KEY",
    "SUPABASE_URL",
    "MONGODB_URI",
    "QDRANT_URL",
    "QDRANT_API_KEY",
)

if IN_COLAB:
    from google.colab import userdata

    for var in REQUIRED_ENV_VARS:
        os.environ[var] = userdata.get(var)
else:
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass

    missing = [k for k in REQUIRED_ENV_VARS if not os.environ.get(k)]

    if missing:
        raise RuntimeError(
            f"Missing required environment variable(s): {missing}. "
            "Set them in your shell (export SUPABASE_URL=...) or in a .env file "
            "in the notebook's working directory before re-running this cell."
        )

for var in REQUIRED_ENV_VARS:
    print(f"{var} set:", bool(os.environ.get(var)))

HF_TOKEN set: True
OPENAI_API_KEY set: True
SUPABASE_URL set: True
MONGODB_URI set: True
QDRANT_URL set: True
QDRANT_API_KEY set: True


## Configuration
Single place to control which databases, workloads, and content types this run
covers. `WORKLOAD_MODEL` fixes each workload's embedding model per the abstracts-
slice evidence in the embedding model benchmark; this is the control variable held
constant across all three databases for a given workload.


In [5]:
HF_REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"

DATABASES = ["pgvector", "mongodb", "qdrant"]

# Fixed embedding model per workload, chosen from the embedding model
# benchmark's abstracts-slice results. Held constant across all three
# databases and both content types for that workload.
WORKLOAD_MODEL = {
    "w1": "intfloat/e5-base-v2",
    "w2": "text-embedding-3-small",
    "w3": "text-embedding-3-large",
}

EMBEDDING_DIMS = {
    "intfloat/e5-base-v2": 768,
    "text-embedding-3-small": 1536,
    "text-embedding-3-large": 3072,
}

CATEGORIES = ["cs.AI", "cs.CL", "cs.CV", "cs.CR", "cs.DB", "cs.DC", "cs.IR", "cs.LG", "cs.NE", "cs.SE"]

WORKLOADS = ["w1", "w2", "w3"]
CONTENT_TYPES = ["abstracts", "pdf_chunks"]

METRICS = [
    "mrr",
    "recall@1",
    "recall@5",
    "ndcg@5",
    "ndcg@10",
]

TOP_K = max(int(m.split("@")[1]) for m in METRICS if "@" in m)

LOCAL_DATA_ROOT = Path("data")
OUTPUT_DIR = Path("db_benchmark_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# HF paths for resumable per-unit results. One JSON file per
# (database, workload, content_type, category) unit.
HF_RESULTS_PREFIX = "db_benchmark_results"
HF_RUNS_PREFIX = "db_benchmark_runs"  # optional raw Run objects for audit

## Download data from Hugging Face
Pulls the chunk embeddings for the three workload-fixed models only (not all four
models from the embedding benchmark) and the query CSVs, same `snapshot_download`
pattern as the embedding model benchmark.


In [6]:
print("TEST: Jupyter is working")

TEST: Jupyter is working


In [7]:
import os

print("HF:", bool(os.environ.get("HF_TOKEN")))
print("OpenAI:", bool(os.environ.get("OPENAI_API_KEY")))
print("PostgreSQL:", bool(os.environ.get("SUPABASE_URL")))
print("MongoDB:", bool(os.environ.get("MONGODB_URI")))
print("Qdrant:", bool(os.environ.get("QDRANT_URL")))
print("Qdrant key:", bool(os.environ.get("QDRANT_API_KEY")))

HF: True
OpenAI: True
PostgreSQL: True
MongoDB: True
Qdrant: True
Qdrant key: True


In [8]:
print("Starting environment check...")

required = [
    "HF_TOKEN",
    "OPENAI_API_KEY",
    "SUPABASE_URL",
    "MONGODB_URI",
    "QDRANT_URL",
    "QDRANT_API_KEY",
]

missing = [k for k in required if not os.environ.get(k)]

print("Missing:", missing)
print("Environment check complete.")

Starting environment check...
Missing: []
Environment check complete.


In [9]:
needed_models = set(WORKLOAD_MODEL.values())

allow_patterns = [f"embeddings/{model}/*" for model in needed_models]
allow_patterns.append("question_answer_pairs/*")

snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DATA_ROOT,
    allow_patterns=allow_patterns,
)

print("Download complete. Local tree:")

for p in sorted(LOCAL_DATA_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(LOCAL_DATA_ROOT))

Fetching 122 files:   0%|          | 0/122 [00:00<?, ?it/s]

Download complete. Local tree:
  .cache\huggingface\.gitignore
  .cache\huggingface\CACHEDIR.TAG
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.AI.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.CL.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.CR.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.CV.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.DB.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.DC.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.IR.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.LG.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.NE.jsonl.metadata
  .cache\huggingface\download\embeddings\intfloat\e5-base-v2\abstracts\cs.S

## Resumability ledger
Before doing any work, check the HF repo for which (database, workload,
content_type, category) units already have a results file. Re-running this
notebook after any interruption, a crashed kernel, a network drop mid-ingestion,
a rate limit, skips every unit already recorded and resumes exactly where it left
off. This is the same resumable-ledger pattern used in the corpus harvesting and
PDF extraction stages of the pipeline (persistent identifier ledgers, hash
indexes), applied here at the evaluation-unit level.


In [10]:
hf_api = HfApi()


def unit_key(database: str, workload: str, content_type: str, category: str) -> str:
    return f"{database}__{workload}__{content_type}__{category}"


def result_path_on_hf(database: str, workload: str, content_type: str, category: str) -> str:
    return f"{HF_RESULTS_PREFIX}/{database}/{workload}/{content_type}/{category}.json"


def load_completed_units() -> set[str]:
    """
    Returns the set of unit_key(...) strings already present on HF.

    A unit is considered complete if and only if its results JSON exists on
    the HF dataset repo. Ingestion is NOT separately tracked here: if a
    result file exists, both ingestion and querying for that unit already
    happened successfully, since the result file is only written after a
    successful query pass (see the save step below).
    """
    try:
        repo_files = hf_api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
    except Exception as e:
        print(f"  WARNING: could not list repo files ({e}); assuming nothing is complete")
        return set()

    completed = set()

    for database in DATABASES:
        for workload in WORKLOADS:
            for content_type in CONTENT_TYPES:
                for category in CATEGORIES:
                    path = result_path_on_hf(database, workload, content_type, category)
                    if path in repo_files:
                        completed.add(unit_key(database, workload, content_type, category))

    return completed


COMPLETED_UNITS = load_completed_units()

print(f"{len(COMPLETED_UNITS)} unit(s) already complete on HF; these will be skipped.")

0 unit(s) already complete on HF; these will be skipped.


## Load chunk embeddings (per workload's fixed model)
Same JSONL format as the embedding model benchmark:
`embeddings/{model}/{content_type}/{category}.jsonl`, one JSON object per line,
`{"chunk_id": ..., "embedding": [...]}`. Loaded per (workload, content_type,
category) rather than all at once, since ingestion proceeds one unit at a time
for resumability.


In [11]:
def load_chunk_embeddings_for_unit(
    model_name: str,
    content_type: str,
    category: str,
) -> dict[str, np.ndarray]:
    """Returns {chunk_id: embedding_vector} for one model/content_type/category."""

    file_path = (
        LOCAL_DATA_ROOT
        / "embeddings"
        / model_name
        / content_type
        / f"{category}.jsonl"
    )

    embeddings: dict[str, np.ndarray] = {}

    if not file_path.exists():
        print(f"  WARNING: missing {file_path}")
        return embeddings

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)
            embeddings[str(record["chunk_id"])] = np.array(
                record["embedding"],
                dtype=np.float32,
            )

    return embeddings


def _normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """L2-normalize float32 embeddings for cosine similarity."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return embeddings / norms

## Load queries
Identical loading and repair logic to the embedding model benchmark, reused
verbatim since the query CSVs themselves have not changed. The three workloads
have different CSV schemas reflecting genuinely different task structures:

- **W1** (`chunk_id, category, bertopic_cluster, question, gold_chunk_id`):
  short-context factual lookup, one query maps to exactly one correct chunk.
- **W2** (`pair_id, category, pair_type, chunk_id_a, chunk_id_b, question`):
  multi-hop reasoning, one query maps to two correct chunks.
- **W3** (`chunk_id, category, bertopic_cluster, query`): semantic similarity,
  one query maps to one anchor chunk.


In [12]:
def load_and_repair_csv(path: Path, expected_columns: list[str]) -> pd.DataFrame:
    """Loads a CSV and strips out phantom Unnamed columns / fully-blank rows."""
    df = pd.read_csv(path)

    extra_cols = [c for c in df.columns if c not in expected_columns]
    if extra_cols:
        df = df[[c for c in expected_columns if c in df.columns]]

    df = df.dropna(how="all")

    return df.reset_index(drop=True)


EXPECTED_COLUMNS_BY_WORKLOAD = {
    "w1": ["chunk_id", "category", "bertopic_cluster", "question", "gold_chunk_id"],
    "w2": ["pair_id", "category", "pair_type", "chunk_id_a", "chunk_id_b", "question"],
    "w3": ["chunk_id", "category", "bertopic_cluster", "query"],
}


def load_queries_for_category(
    content_type: str,
    workload: str,
    category: str,
) -> pd.DataFrame | None:
    """Load and normalize queries for ONE category, unlike the embedding
    benchmark which concatenated all categories. Category-level granularity
    is needed here to match the (database, workload, content_type, category)
    resumability unit."""

    folder = LOCAL_DATA_ROOT / "question_answer_pairs" / content_type / workload

    matching_paths = [p for p in folder.glob("*.csv") if category in p.name]

    if not matching_paths:
        return None

    dfs = [
        load_and_repair_csv(p, EXPECTED_COLUMNS_BY_WORKLOAD[workload])
        for p in matching_paths
    ]

    combined = pd.concat(dfs, ignore_index=True)

    if "category" not in combined.columns:
        combined["category"] = category

    if workload == "w1":
        combined["query_id"] = [f"w1_{category}_{i}" for i in range(len(combined))]
        combined["query_text"] = combined["question"]
        combined["relevant_chunk_ids"] = combined["gold_chunk_id"].apply(lambda x: [str(x)])

    elif workload == "w2":
        combined["query_id"] = [f"w2_{category}_{i}" for i in range(len(combined))]
        combined["query_text"] = combined["question"]
        combined["relevant_chunk_ids"] = combined.apply(
            lambda row: [str(row["chunk_id_a"]), str(row["chunk_id_b"])],
            axis=1,
        )

    elif workload == "w3":
        combined["query_id"] = [f"w3_{category}_{i}" for i in range(len(combined))]
        combined["query_text"] = combined["query"]
        combined["relevant_chunk_ids"] = combined["chunk_id"].apply(lambda x: [str(x)])

    return combined[["query_id", "query_text", "relevant_chunk_ids", "category"]]

## Filtering pdf_chunks queries to the available corpus
`pdf_chunks` embeddings for the workload-fixed models were generated capped at
roughly the first 100 papers per category (see the generation notebook), not the
full PDF corpus. Queries whose gold chunk(s) fall outside this capped set cannot
be answered correctly by any database regardless of retrieval quality, so they are
dropped before scoring, exactly as in the embedding model benchmark.

Unlike the embedding model benchmark, which intersected chunk IDs across four
candidate models, this filter only needs to check availability against ONE model
per workload, since the embedding model is already fixed per workload here.


In [13]:
def filter_queries_to_available_chunks(
    queries_df: pd.DataFrame,
    available_chunk_ids: set[str],
) -> pd.DataFrame:
    """Keep only queries for which every gold chunk is available."""

    mask = queries_df["relevant_chunk_ids"].apply(
        lambda ids: all(cid in available_chunk_ids for cid in ids)
    )

    return queries_df.loc[mask].reset_index(drop=True)


def load_queries_for_unit(
    workload: str,
    content_type: str,
    category: str,
    chunk_embeddings: dict[str, np.ndarray],
) -> pd.DataFrame | None:
    queries_df = load_queries_for_category(content_type, workload, category)

    if queries_df is None or queries_df.empty:
        return queries_df

    if content_type != "pdf_chunks":
        return queries_df

    available_chunk_ids = set(chunk_embeddings.keys())

    filtered_df = filter_queries_to_available_chunks(queries_df, available_chunk_ids)

    dropped = len(queries_df) - len(filtered_df)

    if dropped:
        print(
            f"    dropped {dropped} of {len(queries_df)} queries whose "
            f"gold chunk(s) fall outside the capped pdf_chunks corpus"
        )

    return filtered_df

## Query embedding
Same on-the-fly query embedding as the embedding model benchmark, but now only
ever called with ONE fixed model per workload, not all four candidates. Latency
is measured separately here (`embedding_time_sec`) so it can be reported apart
from the database query round-trip time (`db_query_time_sec`), since the two are
governed by different systems (OpenAI/local inference vs. the database engine)
and conflating them would misattribute latency to the wrong layer.


In [14]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer

openai_client = OpenAI()

_st_model_cache: dict[str, SentenceTransformer] = {}


def embed_queries_timed(
    model_name: str,
    texts: list[str],
) -> tuple[np.ndarray, list[float]]:
    """
    Returns (embeddings, per_query_embedding_time_sec).

    OpenAI models are batched in a single API call, so the batch latency is
    divided evenly across the batch as an approximation of per-query cost.
    The open-source model is timed per-query individually since it is a
    local forward pass and can be measured directly without a network
    round-trip confound.
    """

    if model_name.startswith("text-embedding-"):
        start = time.perf_counter()
        response = openai_client.embeddings.create(model=model_name, input=texts)
        elapsed = time.perf_counter() - start

        per_query_time = [elapsed / len(texts)] * len(texts)

        embeddings = np.array([item.embedding for item in response.data], dtype=np.float32)
        return embeddings, per_query_time

    else:
        if model_name not in _st_model_cache:
            _st_model_cache[model_name] = SentenceTransformer(model_name)

        model = _st_model_cache[model_name]

        per_query_time = []
        vectors = []

        for text in texts:
            start = time.perf_counter()
            vec = model.encode([text], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
            per_query_time.append(time.perf_counter() - start)
            vectors.append(vec[0])

        return np.asarray(vectors, dtype=np.float32), per_query_time

## Database client interface
A single, uniform interface is implemented for all three paradigms so that the
evaluation loop below can treat pgvector, MongoDB, and Qdrant identically. This
mirrors the project's broader preference for pure Python clients over an
abstraction layer such as LangChain: `psycopg2` for pgvector, `pymongo` for
MongoDB Atlas Vector Search, and `qdrant-client` for Qdrant, each used directly
so that no added abstraction latency pollutes the timing measurements this
benchmark depends on.

Each client exposes exactly two operations:

- `upsert_chunks(collection_name, chunk_ids, embeddings, dim)` — idempotent
  insert, skips (database-side) chunks that already exist so re-running
  ingestion after an interruption never double-inserts.
- `query_top_k(collection_name, query_embedding, k, dim=None)` — returns
  `(chunk_ids_ranked, scores_ranked, db_query_time_sec)`. `dim` is only
  consumed by the pgvector client (see below); MongoDB and Qdrant accept and
  ignore it, kept purely so the call site can pass `dim=` uniformly to all
  three clients without a database-specific branch.

Collections/tables/indexes are namespaced per (workload, content_type) so that
the three workload-fixed models never collide inside the same database, since
`text-embedding-3-small` and `intfloat/e5-base-v2` have different dimensionality
and cannot share a vector index.

**pgvector's 2000-dimension index limit.** pgvector's plain `vector` type can
only be indexed (HNSW or IVFFlat, both) up to 2000 dimensions. W3's
`text-embedding-3-large` is 3072-D, which exceeds this. The fix is NOT to skip
indexing for W3, since an unindexed `ORDER BY embedding <=> ...` query falls
back to a full sequential scan, computing cosine distance against every row in
the table on every single query. Instead, the pgvector client below builds an
HNSW index on a `halfvec` cast of the embedding column whenever `dim > 2000`.
`halfvec` (half-precision floats, pgvector >= 0.7.0) supports HNSW indexing up
to 4000 dimensions, comfortably covering W3's 3072-D vectors, without needing
a separate table or duplicated storage. The query side must cast its query
vector to the same `halfvec` expression the index was built on, or Postgres
silently reverts to a sequential scan even though the index exists.


In [ ]:
import psycopg2
from psycopg2.extras import execute_values


def collection_name(workload: str, content_type: str) -> str:
    return f"{workload}_{content_type}".replace(".", "_")


# pgvector's plain `vector` type can only be indexed (HNSW or IVFFlat) up to
# 2000 dimensions - this is a hard limit in pgvector itself, not specific to
# HNSW. `text-embedding-3-large` is 3072-D, so a plain `vector` HNSW index on
# it is impossible, not just slow.
#
# The fix is NOT to skip indexing above 2000-D. Skipping means every W3 query
# falls back to a full sequential scan (`ORDER BY embedding <=> ...` with no
# index), which computes cosine distance against every row in the table on
# every query. That is the slowdown you were seeing.
#
# The fix is pgvector's `halfvec` type (half-precision floats, available
# since pgvector 0.7.0), which raises the indexable limit to 4000 dimensions.
# An HNSW index is built on a `halfvec` CAST of the same `vector` column, so
# no duplicate storage or separate table is needed. The query itself must
# cast its query vector to the SAME halfvec expression the index was built
# on, or Postgres will not use the index (see `query_top_k` below).
PGVECTOR_HNSW_MAX_DIM = 2000
PGVECTOR_HALFVEC_HNSW_MAX_DIM = 4000


class PgVectorClient:
    def __init__(self, dsn: str):
        self.conn = psycopg2.connect(dsn)
        self.conn.autocommit = True

        with self.conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    def _ensure_table(self, table: str, dim: int) -> None:
        with self.conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

            cur.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {table} (
                    chunk_id TEXT PRIMARY KEY,
                    embedding vector({dim})
                );
                """
            )

            if dim <= PGVECTOR_HNSW_MAX_DIM:
                # Plain vector HNSW index. Covers W1 (768-D) and W2 (1536-D).
                cur.execute(
                    f"""
                    CREATE INDEX IF NOT EXISTS {table}_hnsw_idx
                    ON {table} USING hnsw (embedding vector_cosine_ops);
                    """
                )
            elif dim <= PGVECTOR_HALFVEC_HNSW_MAX_DIM:
                # halfvec-cast HNSW index. Covers W3 (3072-D). The cast
                # expression here must exactly match the cast used in
                # query_top_k's ORDER BY, or Postgres silently falls back
                # to a sequential scan instead of using this index.
                cur.execute(
                    f"""
                    CREATE INDEX IF NOT EXISTS {table}_hnsw_halfvec_idx
                    ON {table} USING hnsw ((embedding::halfvec({dim})) halfvec_cosine_ops);
                    """
                )
            else:
                # dim > 4000: no pgvector index type can cover this even
                # with halfvec. Not reached by any workload in this study
                # (max is 3072), but left explicit rather than silently
                # falling back to an unindexed sequential scan.
                raise ValueError(
                    f"dim={dim} exceeds pgvector's halfvec HNSW limit of "
                    f"{PGVECTOR_HALFVEC_HNSW_MAX_DIM}; no index type can "
                    f"cover this dimensionality."
                )

        self.conn.commit()

    def upsert_chunks(
        self,
        table: str,
        chunk_ids: list[str],
        embeddings: np.ndarray,
        dim: int,
    ) -> None:
        self._ensure_table(table, dim)

        rows = [
            (cid, list(map(float, vec)))
            for cid, vec in zip(chunk_ids, embeddings)
        ]

        with self.conn.cursor() as cur:
            execute_values(
                cur,
                f"""
                INSERT INTO {table} (chunk_id, embedding)
                VALUES %s
                ON CONFLICT (chunk_id) DO NOTHING;
                """,
                rows,
                template="(%s, %s)",
            )

    def query_top_k(
        self,
        table: str,
        query_embedding: np.ndarray,
        k: int,
        dim: int | None = None,
    ) -> tuple[list[str], list[float], float]:
        if dim is None:
            dim = len(query_embedding)

        vec_literal = list(map(float, query_embedding))

        start = time.perf_counter()
        with self.conn.cursor() as cur:
            if dim <= PGVECTOR_HNSW_MAX_DIM:
                # Uses the plain vector HNSW index built above.
                cur.execute(
                    f"""
                    SELECT chunk_id, 1 - (embedding <=> %s::vector) AS score
                    FROM {table}
                    ORDER BY embedding <=> %s::vector
                    LIMIT %s;
                    """,
                    (vec_literal, vec_literal, k),
                )
            else:
                # Must cast the query vector to halfvec too, matching the
                # index expression exactly, or this query silently reverts
                # to a full sequential scan even though the halfvec index
                # exists on the table.
                cur.execute(
                    f"""
                    SELECT chunk_id,
                           1 - (embedding::halfvec({dim}) <=> %s::halfvec({dim})) AS score
                    FROM {table}
                    ORDER BY embedding::halfvec({dim}) <=> %s::halfvec({dim})
                    LIMIT %s;
                    """,
                    (vec_literal, vec_literal, k),
                )

            rows = cur.fetchall()

        elapsed = time.perf_counter() - start

        chunk_ids = [r[0] for r in rows]
        scores = [float(r[1]) for r in rows]

        return chunk_ids, scores, elapsed

    def close(self) -> None:
        self.conn.close()


In [ ]:
from pymongo import MongoClient, UpdateOne


class MongoVectorClient:
    def __init__(self, uri: str, db_name: str = "rag_benchmark"):
        self.client = MongoClient(uri)
        self.db = self.client[db_name]

    def _collection(self, name: str):
        return self.db[name]

    def _ensure_vector_index(self, name: str, dim: int) -> None:
        coll = self._collection(name)
        existing = [ix["name"] for ix in coll.list_search_indexes()] if hasattr(coll, "list_search_indexes") else []

        index_name = f"{name}_vector_idx"

        if index_name in existing:
            return

        try:
            coll.create_search_index(
                {
                    "name": index_name,
                    "type": "vectorSearch",
                    "definition": {
                        "fields": [
                            {
                                "type": "vector",
                                "path": "embedding",
                                "numDimensions": dim,
                                "similarity": "cosine",
                            }
                        ]
                    },
                }
            )
        except Exception as e:
            print(f"  NOTE: search index creation for {name} raised ({e}); "
                  f"it may already exist or may need to be created manually "
                  f"in the Atlas UI, since Atlas Search index creation via "
                  f"driver requires an Atlas cluster tier that supports it.")

    def upsert_chunks(
        self,
        name: str,
        chunk_ids: list[str],
        embeddings: np.ndarray,
        dim: int,
    ) -> None:
        self._ensure_vector_index(name, dim)

        coll = self._collection(name)

        operations = [
            UpdateOne(
                {"_id": cid},
                {"$setOnInsert": {"_id": cid, "embedding": list(map(float, vec))}},
                upsert=True,
            )
            for cid, vec in zip(chunk_ids, embeddings)
        ]

        if operations:
            coll.bulk_write(operations, ordered=False)

    def query_top_k(
        self,
        name: str,
        query_embedding: np.ndarray,
        k: int,
        dim: int | None = None,  # unused; accepted for a uniform client interface
    ) -> tuple[list[str], list[float], float]:
        coll = self._collection(name)
        index_name = f"{name}_vector_idx"

        pipeline = [
            {
                "$vectorSearch": {
                    "index": index_name,
                    "path": "embedding",
                    "queryVector": list(map(float, query_embedding)),
                    "numCandidates": max(k * 10, 100),
                    "limit": k,
                }
            },
            {"$project": {"_id": 1, "score": {"$meta": "vectorSearchScore"}}},
        ]

        start = time.perf_counter()
        results = list(coll.aggregate(pipeline))
        elapsed = time.perf_counter() - start

        chunk_ids = [r["_id"] for r in results]
        scores = [float(r["score"]) for r in results]

        return chunk_ids, scores, elapsed

    def close(self) -> None:
        self.client.close()

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


class QdrantVectorClient:
    def __init__(self, url: str, api_key: str):
        self.client = QdrantClient(url=url, api_key=api_key)

    def _ensure_collection(self, name: str, dim: int) -> None:
        existing = [c.name for c in self.client.get_collections().collections]

        if name in existing:
            return

        self.client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
        )

    def upsert_chunks(
        self,
        name: str,
        chunk_ids: list[str],
        embeddings: np.ndarray,
        dim: int,
    ) -> None:
        self._ensure_collection(name, dim)

        points = [
            PointStruct(id=abs(hash(cid)) % (10 ** 12), vector=list(map(float, vec)), payload={"chunk_id": cid})
            for cid, vec in zip(chunk_ids, embeddings)
        ]

        if points:
            self.client.upsert(collection_name=name, points=points, wait=True)

    def query_top_k(
        self,
        name: str,
        query_embedding: np.ndarray,
        k: int,
        dim: int | None = None,  # unused; accepted for a uniform client interface
    ) -> tuple[list[str], list[float], float]:
        start = time.perf_counter()
        results = self.client.search(
            collection_name=name,
            query_vector=list(map(float, query_embedding)),
            limit=k,
        )
        elapsed = time.perf_counter() - start

        chunk_ids = [r.payload["chunk_id"] for r in results]
        scores = [float(r.score) for r in results]

        return chunk_ids, scores, elapsed

    def close(self) -> None:
        pass

## Client factory
Lazily connects to each database only when its turn comes in the evaluation loop,
so a credential problem with one database does not block the other two from
running.


In [18]:
def get_client(database: str):
    if database == "pgvector":
        return PgVectorClient(os.environ["SUPABASE_URL"])
    elif database == "mongodb":
        return MongoVectorClient(os.environ["MONGODB_URI"])
    elif database == "qdrant":
        return QdrantVectorClient(os.environ["QDRANT_URL"], os.environ["QDRANT_API_KEY"])
    else:
        raise ValueError(f"Unknown database: {database}")

## Qrels and metric scoring
Same `ranx` pattern as the embedding model benchmark: `Qrels` is the ground-truth
relevance mapping (with W2 carrying two relevant chunks per query), and a `Run` is
built from each database's ranked retrieval output. Latency is captured
separately, per query, alongside the ranking data, since latency is itself part of
what this benchmark is meant to report (Recommendation for a Q1 systems paper: a
paradigm that wins on retrieval quality but is an order of magnitude slower is not
a strictly better choice).


In [ ]:
def create_qrels(queries_df: pd.DataFrame) -> Qrels:
    qrels_dict = {}

    for _, row in queries_df.iterrows():
        qrels_dict[row["query_id"]] = {cid: 1 for cid in row["relevant_chunk_ids"]}

    return Qrels(qrels_dict)


def build_run_from_db(
    client,
    table_name: str,
    queries_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    embedding_times: list[float],
    top_k: int,
    dim: int,
) -> tuple[Run, list[dict]]:
    """
    Queries the database once per query, builds a ranx Run, and returns
    per-query audit rows containing BOTH timing/latency AND the ranked
    chunk_ids/scores actually retrieved. Saving the ranking data too, not
    just timing, is what makes the later cross-database significance
    comparison reconstructible purely from the saved HF files, without
    needing to re-query any database.
    """

    run_dict = {}
    audit_rows = []

    for i, (_, row) in enumerate(queries_df.iterrows()):
        query_id = row["query_id"]

        chunk_ids, scores, db_query_time_sec = client.query_top_k(
            table_name,
            query_embeddings[i],
            top_k,
            dim=dim,
        )

        run_dict[query_id] = {cid: score for cid, score in zip(chunk_ids, scores)}

        audit_rows.append(
            {
                "query_id": query_id,
                "retrieved_chunk_ids": chunk_ids,
                "retrieved_scores": scores,
                "embedding_time_sec": embedding_times[i],
                "db_query_time_sec": db_query_time_sec,
                "total_time_sec": embedding_times[i] + db_query_time_sec,
            }
        )

    return Run(run_dict, name=table_name), audit_rows

## Report formatting helper
Note this is NOT the same reporting pattern as the embedding model benchmark.
There, `compare()` was called with all four candidate models at once, so
letter-coded significance markers (Fisher's exact test between model pairs)
were meaningful and printed. Here, each `compare()` call evaluates exactly ONE
database's Run per unit, since ingestion and querying happen one database at a
time for resumability. `ranx`'s pairwise significance testing needs at least
two Runs in the same `compare()` call to produce a comparison, so it does not
apply to these single-Run calls.

Cross-database significance testing is instead done afterward, once, in the
aggregation section below, by loading all three databases' final per-slice
scores back from HF and running `ranx.compare()` across all three Runs
together for each slice. This keeps the resumable ingestion/query loop simple
while still producing a proper three-way significance comparison for the
paper.

This helper therefore just prints each database's own scores plainly, using
`ranx`'s built-in table formatter.


In [20]:
def print_single_run_report(report: Report, database: str) -> None:
    report.rounding_digits = 4

    console = Console()
    table = Table(box=HEAVY, show_lines=False, title=f"{database}")

    table.add_column("Metric", style="bold")
    table.add_column("Score")

    model_name = report.model_names[0]

    for metric in report.metrics:
        score = report.results[model_name][metric]
        table.add_row(metric.upper(), f"{score:.4f}")

    console.print(table)

## Save-and-upload helper
Writes the unit's results (metrics + raw per-query timing rows) to local disk,
then immediately pushes both to the private HF dataset. This upload happens
right after each unit finishes, not batched at the end of the notebook, so that
an interruption anywhere after this point still leaves every prior unit safely
recorded on HF and resumable on the next run.


In [21]:
def save_and_upload_unit(
    database: str,
    workload: str,
    content_type: str,
    category: str,
    metrics_row: dict,
    audit_rows: list[dict],
) -> None:
    local_results_dir = OUTPUT_DIR / "results" / database / workload / content_type
    local_results_dir.mkdir(parents=True, exist_ok=True)

    local_runs_dir = OUTPUT_DIR / "runs" / database / workload / content_type
    local_runs_dir.mkdir(parents=True, exist_ok=True)

    results_path = local_results_dir / f"{category}.json"
    runs_path = local_runs_dir / f"{category}.json"

    with open(results_path, "w") as f:
        json.dump(metrics_row, f, indent=2)

    with open(runs_path, "w") as f:
        json.dump(audit_rows, f, indent=2)

    # Mark this unit as completed locally.
    # We do not upload to the original Hugging Face dataset because
    # we do not have write permission to that repository.
    COMPLETED_UNITS.add(unit_key(database, workload, content_type, category))

    print(
        f"    Saved results + timing locally for "
        f"{database}/{workload}/{content_type}/{category}"
    )
    print(f"    Results: {results_path}")
    print(f"    Runs:    {runs_path}")

## Run all evaluation units
Outer-to-inner loop order: database -> workload -> content_type -> category.
For every unit already recorded on HF (per `COMPLETED_UNITS`), the unit is
skipped entirely, including its ingestion step, so re-running this notebook
after an interruption does not re-embed, re-ingest, or re-query anything already
done.

For each new unit:

1. Load that workload's fixed-model chunk embeddings for this category.
2. Ingest them into the current database (idempotent upsert).
3. Load and filter this category's queries for this workload/content_type.
4. Embed the queries with the workload's fixed model (timed).
5. Query the database for each query's top-k results (timed).
6. Score with `ranx` against the Qrels.
7. Save + upload results and raw timing immediately.


In [ ]:
for database in DATABASES:
    print(f"\n{'#' * 70}")
    print(f"  DATABASE: {database.upper()}")
    print(f"{'#' * 70}")

    client = None

    try:
        for workload in WORKLOADS:
            model_name = WORKLOAD_MODEL[workload]
            dim = EMBEDDING_DIMS[model_name]

            for content_type in CONTENT_TYPES:
                table_name = collection_name(workload, content_type)

                for category in CATEGORIES:
                    key = unit_key(database, workload, content_type, category)

                    if key in COMPLETED_UNITS:
                        print(f"  [SKIP] {key} already complete on HF")
                        continue

                    print(f"\n  {'-' * 60}")
                    print(f"  {database} | {workload} | {content_type} | {category} | model={model_name}")
                    print(f"  {'-' * 60}")

                    # Step 1: load this category's fixed-model chunk embeddings.
                    chunk_embeddings = load_chunk_embeddings_for_unit(model_name, content_type, category)

                    if not chunk_embeddings:
                        print(f"    No chunk embeddings available; skipping.")
                        continue

                    # Step 2: lazily connect and ingest.
                    if client is None:
                        client = get_client(database)

                    chunk_ids_list = list(chunk_embeddings.keys())
                    chunk_matrix = np.stack([chunk_embeddings[cid] for cid in chunk_ids_list])

                    client.upsert_chunks(table_name, chunk_ids_list, chunk_matrix, dim)

                    print(f"    Ingested {len(chunk_ids_list)} chunks into {database}.{table_name}")

                    # Step 3: load + filter queries for this unit.
                    queries_df = load_queries_for_unit(
                        workload, content_type, category, chunk_embeddings
                    )

                    if queries_df is None or queries_df.empty:
                        print(f"    No queries available; skipping.")
                        del chunk_embeddings, chunk_matrix
                        gc.collect()
                        continue

                    print(f"    {len(queries_df)} queries loaded")

                    # Step 4: embed queries (timed).
                    query_texts = queries_df["query_text"].tolist()
                    query_embeddings, embedding_times = embed_queries_timed(model_name, query_texts)
                    query_embeddings = _normalize_embeddings(query_embeddings)

                    # Step 5 + 6: query the DB, build Run, score.
                    qrels = create_qrels(queries_df)

                    run, audit_rows = build_run_from_db(
                        client, table_name, queries_df, query_embeddings, embedding_times, TOP_K, dim
                    )

                    report = compare(qrels=qrels, runs=[run], metrics=METRICS, max_p=0.05, stat_test="fisher")

                    metrics_row = {
                        "database": database,
                        "workload": workload,
                        "content_type": content_type,
                        "category": category,
                        "model": model_name,
                        "n_queries": len(queries_df),
                        "n_chunks_ingested": len(chunk_ids_list),
                        **{metric: report.results[table_name][metric] for metric in METRICS},
                        "mean_embedding_time_sec": float(np.mean(embedding_times)),
                        "mean_db_query_time_sec": float(np.mean([r["db_query_time_sec"] for r in audit_rows])),
                        "mean_total_time_sec": float(np.mean([r["total_time_sec"] for r in audit_rows])),
                    }

                    print_single_run_report(report, database)
                    print(f"    mean_db_query_time={metrics_row['mean_db_query_time_sec']*1000:.1f}ms  "
                          f"mean_embedding_time={metrics_row['mean_embedding_time_sec']*1000:.1f}ms")

                    # Step 7: save + upload immediately.
                    save_and_upload_unit(database, workload, content_type, category, metrics_row, audit_rows)

                    del chunk_embeddings, chunk_matrix, query_embeddings, run, qrels, report
                    gc.collect()

    finally:
        if client is not None:
            client.close()
            print(f"\n  Closed connection to {database}.")

print("\nAll units complete (or already were).")

In [30]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.DB.csv")

with open(path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    for line_no, row in enumerate(reader, start=1):
        if len(row) != 6:
            print(f"BAD LINE {line_no}: {len(row)} fields")
            print(row)

BAD LINE 12: 7 fields
['cs.DB_w2_010', 'cs.DB', 'same_cluster', '2301.05811_p8', '2301.05811_p11', 'How do the variance bounds derived for the idealized inner-product estimator F* relate to the practical performance differences among Johnson-Lindenstrauss', ' CountSketch and AMS sketches when used for the same estimation task?']
BAD LINE 21: 8 fields
['cs.DB_w2_019', 'cs.DB', 'adjacent_cluster', '2406.10940_p48', '2406.10940_p67', 'How do the trialability and documentation scores collected for commercial data-quality tools relate to the concrete feature lists (profiling', ' anomaly detection', ' lineage) that appear in the later appendix tables?']
BAD LINE 29: 7 fields
['cs.DB_w2_027', 'cs.DB', 'adjacent_cluster', '2408.04919_p13', '2408.04919_p10', 'What effect does increasing the number of sample values have on EX scores for simple versus challenging text-to-SQL questions', ' and how does that finding interact with the broader LLM-based SQL generation literature?']
BAD LINE 30: 7 fie

In [23]:
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/cs.CR.csv")

print("Exists:", path.exists())
print("Path:", path)

Exists: False
Path: data\question_answer_pairs\pdf_chunks\w2\cs.CR.csv


In [31]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.DB.csv")

with open(path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    rows = list(reader)

fixed_rows = [rows[0]]

for row in rows[1:]:
    if len(row) > 6:
        # First 5 columns are fixed.
        # Everything after that belongs to the question text.
        row = row[:5] + [",".join(row[5:])]
    fixed_rows.append(row)

with open(path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(fixed_rows)

print("CSV fixed successfully.")
print("Total rows:", len(fixed_rows) - 1)

CSV fixed successfully.
Total rows: 50


In [24]:
from pathlib import Path

root = Path("data/question_answer_pairs/pdf_chunks/w2")

print("Folder exists:", root.exists())

if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(p)

Folder exists: True
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.CR.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.CV.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.DB.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.DC.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.IR.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.LG.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.NE.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs.SE.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs_AI.csv
data\question_answer_pairs\pdf_chunks\w2\qna_w2_pdf_chunks_cs_CL.csv
data\question_answer_pairs\pdf_chunks\w2\rename.bat


In [32]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.DB.csv")

bad_rows = []

with open(path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    for line_no, row in enumerate(reader, start=1):
        if len(row) != 6:
            bad_rows.append((line_no, row))

print("Bad rows:", bad_rows)

Bad rows: []


In [25]:
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.CR.csv")

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"LINE {i+1}: {line.rstrip()}")

LINE 1: pair_id,category,pair_type,chunk_id_a,chunk_id_b,question
LINE 2: cs.CR_w2_000,cs.CR,same_cluster,1709.06270_p29,1709.06270_p20,How do the near-ideal information entropy values reported for the cipher images of both Lena and the all-zero image support the claim that the combined quasigroup-chaos scheme has a key space large enough to resist brute-force attack?
LINE 3: cs.CR_w2_001,cs.CR,adjacent_cluster,1709.06270_p25,1907.06466_p4,Could the row-column permutation and quasigroup substitution steps shown in the image-cipher block diagram be protected by the ACCESSCONTROL key-enveloping and WRITERSHIELD proxy-file mechanisms when the resulting encrypted images are stored in a multi-user cloud group?
LINE 4: cs.CR_w2_002,cs.CR,adjacent_cluster,1609.09219_p3,1811.03457_p5,How does the two-step GMGK generator that mixes linear-congruence chaotic maps with a shift-register feedback differ from TimeCrypt’s PRG-based key-derivation tree when both are used to produce long keystreams for

In [26]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.CR.csv")

with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    
    for line_no, row in enumerate(reader, start=1):
        if len(row) != 6:
            print(f"BAD LINE {line_no}: {len(row)} fields")
            print(row)

BAD LINE 5: 7 fields
['cs.CR_w2_003', 'cs.CR', 'same_cluster', '1901.06935_p13', '1702.06554_p9', 'What connection exists between the extensive literature on electromagnetic and acoustic interference with sensors and the Bayesian cost assignments used by the P-D detector to distinguish multipath', ' spoofing and jamming in GNSS receivers?']
BAD LINE 9: 7 fields
['cs.CR_w2_007', 'cs.CR', 'same_cluster', '1812.04293_p10', '1808.01546_p9', 'What shared limitation of deterministic defenses against adversarial examples is highlighted by both the Mix’n’Squeeze advantage analysis and the ATMPA transferability results on malware detectors', ' and how does each paper propose to mitigate it?']
BAD LINE 33: 7 fields
['cs.CR_w2_031', 'cs.CR', 'same_cluster', '1905.11880_p20', '1904.13173_p18', 'In what way could the permanent', ' content-addressed storage provided by IPFS be leveraged by an argumentation-based attribution reasoner that needs to retain immutable evidence of past cyber-attacks?']
BA

In [27]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.CR.csv")

with open(path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    rows = list(reader)

fixed_rows = [rows[0]]  # header

for row in rows[1:]:
    if len(row) > 6:
        # Columns 0-4 are correct; everything else belongs to the question
        row = row[:5] + [",".join(row[5:])]
    fixed_rows.append(row)

with open(path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(fixed_rows)

print("CSV fixed successfully.")
print("Total rows:", len(fixed_rows) - 1)

CSV fixed successfully.
Total rows: 50


In [28]:
import csv
from pathlib import Path

path = Path("data/question_answer_pairs/pdf_chunks/w2/qna_w2_pdf_chunks_cs.CR.csv")

with open(path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    bad = [(i, len(row)) for i, row in enumerate(reader, start=1) if len(row) != 6]

print("Bad rows:", bad)

Bad rows: []


## Aggregate results from HF
Pulls every unit's results JSON back down from HF (rather than relying on
in-memory state, which may not exist if this cell is run in a fresh session
after a previous run already completed ingestion/querying) and assembles the
combined results table, mirroring the embedding model benchmark's aggregation
step.


In [ ]:
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DATA_ROOT,
    allow_patterns=[f"{HF_RESULTS_PREFIX}/*"],
)

all_rows = []

for path in sorted((LOCAL_DATA_ROOT / HF_RESULTS_PREFIX).rglob("*.json")):
    with open(path) as f:
        all_rows.append(json.load(f))

results_df = pd.DataFrame(all_rows)

results_df = (
    results_df
    .sort_values(["database", "workload", "content_type", "category"])
    .reset_index(drop=True)
)

csv_path = OUTPUT_DIR / "db_benchmark_results_by_category.csv"
results_df.to_csv(csv_path, index=False)

print(f"{len(results_df)} unit result rows aggregated -> {csv_path}")
results_df.head()

## Aggregate to (database, workload, content_type) slice level
Category-level rows are averaged (query-count-weighted) up to the same slice
granularity reported in the embedding model benchmark, so the two benchmarks'
result tables are directly comparable in the paper.


In [ ]:
def weighted_mean(df: pd.DataFrame, value_col: str, weight_col: str) -> float:
    return float(np.average(df[value_col], weights=df[weight_col]))


slice_rows = []

for (database, workload, content_type), group in results_df.groupby(["database", "workload", "content_type"]):
    row = {
        "database": database,
        "workload": workload,
        "content_type": content_type,
        "model": group["model"].iloc[0],
        "n_queries": int(group["n_queries"].sum()),
    }

    for metric in METRICS + ["mean_embedding_time_sec", "mean_db_query_time_sec", "mean_total_time_sec"]:
        row[metric] = weighted_mean(group, metric, "n_queries")

    slice_rows.append(row)

slice_df = pd.DataFrame(slice_rows).sort_values(["content_type", "workload", "database"]).reset_index(drop=True)

slice_csv_path = OUTPUT_DIR / "db_benchmark_results_by_slice.csv"
slice_df.to_csv(slice_csv_path, index=False)

print(f"Slice-level results -> {slice_csv_path}")
slice_df

## Cross-database significance testing
Reconstructs each database's per-query `Run` for a given (workload,
content_type) slice from the saved audit JSON files on HF (the
`retrieved_chunk_ids`/`retrieved_scores` fields saved above), rebuilds the
`Qrels` for that slice from the query CSVs, and calls `ranx.compare()` with
all three databases' Runs together. This is the point where Fisher's exact
test at p < 0.05 actually produces a three-way, letter-coded significance
comparison, mirroring the embedding model benchmark's reporting pattern, since
`compare()` needs multiple Runs in one call to compare them against each
other. This step is read-only against already-saved data and does not touch
any live database connection, so it can be re-run freely without re-querying
anything.


In [ ]:
def load_audit_rows(database: str, workload: str, content_type: str, category: str) -> list[dict] | None:
    local_path = OUTPUT_DIR / "runs" / database / workload / content_type / f"{category}.json"

    if local_path.exists():
        with open(local_path) as f:
            return json.load(f)

    # Fall back to HF if not present locally (e.g. fresh session).
    try:
        downloaded = hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            filename=f"{HF_RUNS_PREFIX}/{database}/{workload}/{content_type}/{category}.json",
            local_dir=LOCAL_DATA_ROOT,
        )
        with open(downloaded) as f:
            return json.load(f)
    except Exception:
        return None


def reconstruct_run_for_slice(database: str, workload: str, content_type: str) -> Run | None:
    run_dict = {}

    for category in CATEGORIES:
        audit_rows = load_audit_rows(database, workload, content_type, category)

        if not audit_rows:
            continue

        for row in audit_rows:
            run_dict[row["query_id"]] = {
                cid: score
                for cid, score in zip(row["retrieved_chunk_ids"], row["retrieved_scores"])
            }

    if not run_dict:
        return None

    return Run(run_dict, name=database)


def reconstruct_qrels_for_slice(workload: str, content_type: str) -> Qrels | None:
    qrels_dict = {}

    for category in CATEGORIES:
        queries_df = load_queries_for_category(content_type, workload, category)

        if queries_df is None or queries_df.empty:
            continue

        for _, row in queries_df.iterrows():
            qrels_dict[row["query_id"]] = {cid: 1 for cid in row["relevant_chunk_ids"]}

    if not qrels_dict:
        return None

    return Qrels(qrels_dict)


cross_db_reports = {}

for workload in WORKLOADS:
    for content_type in CONTENT_TYPES:
        qrels = reconstruct_qrels_for_slice(workload, content_type)

        if qrels is None:
            continue

        runs = []

        for database in DATABASES:
            run = reconstruct_run_for_slice(database, workload, content_type)
            if run is not None:
                runs.append(run)

        if len(runs) < 2:
            print(f"  {workload}/{content_type}: fewer than 2 databases have results; skipping comparison.")
            continue

        report = compare(qrels=qrels, runs=runs, metrics=METRICS, max_p=0.05, stat_test="fisher")

        cross_db_reports[(workload, content_type)] = report

        print(f"\n{'=' * 70}")
        print(f"  {workload.upper()} | {content_type}  (all databases)")
        print(f"{'=' * 70}")
        print(report.to_table())

## Next step
`db_benchmark_results_by_slice.csv` is the primary results table for the paper's
Results section (Section V), reporting retrieval quality (MRR, Recall@1,
Recall@5, NDCG@5, NDCG@10) and latency (mean embedding time, mean DB query time,
mean total time) for each of the three database paradigms across all six
workload/content-type slices. `db_benchmark_results_by_category.csv` is the finer-
grained version, useful for a per-category robustness check or an appendix table
if a reviewer asks whether results hold uniformly across arXiv categories.

The raw per-query audit rows, pushed to
`{HF_RUNS_PREFIX}/{database}/{workload}/{content_type}/{category}.json` on HF
alongside the results, contain both the ranked chunk IDs/scores retrieved and the
timing breakdown for every query. These are what to pull if a reviewer or the
supervisor asks for per-query auditability (e.g. "show me exactly which chunk W1
query #42 retrieved from MongoDB and how long it took"), and are also what the
cross-database significance testing section above reconstructs its `Run` objects
from, without needing to re-query any live database.
